# 🚦 RoadRisk AI — 04: Geospatial Hotspot Analysis & DBSCAN Spatial Clustering

**Module Objective:** Discover empirical accident concentrations using Density-Based Spatial Clustering of Applications with Noise (DBSCAN) using the spherical Haversine geodesic metric.

### Topics Covered:
1. **Haversine Geodesic Metric:** Converting degree coordinates to radians and scaling by Earth's radius.
2. **Urban Hotspot Identification:** Clustering incidents in Mumbai, Bangalore, and Delhi.
3. **Noise Identification:** Distinguishing isolated random crashes from high-density recurring corridors.
4. **International Benchmark:** Clustering the UrbanGB Great Britain dataset.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.database import execute_query
from src.hotspot_analysis import compute_dbscan_hotspots, get_city_accident_clusters

print('Geospatial hotspot dependencies loaded successfully.')


## 1. Run DBSCAN Hotspot Clustering for Mumbai
Cluster incidents with neighborhood radius Eps = 1.2 km and MinPts = 15.


In [ ]:
df_clustered, df_summary = get_city_accident_clusters('Mumbai', eps_km=1.2, min_samples=15)

print(f'Total Mumbai Incidents: {len(df_clustered):,}')
print(f'Identified Hotspot Clusters: {len(df_summary)}')
noise_pts = (df_clustered['cluster_label'] == -1).sum()
print(f'Unclustered Noise Points: {noise_pts:,} ({noise_pts/len(df_clustered)*100:.1f}%)\n')

print('Ranked Hotspot Clusters:')
display(df_summary.head())


## 2. Spatial Scatter Plot of Discovered Clusters


In [ ]:
plt.figure(figsize=(9, 7))
# Plot noise in gray
noise = df_clustered[df_clustered['cluster_label'] == -1]
plt.scatter(noise['longitude'], noise['latitude'], c='#555555', s=10, alpha=0.3, label='Noise')

# Plot clustered points with colormap
core = df_clustered[df_clustered['cluster_label'] != -1]
scatter = plt.scatter(core['longitude'], core['latitude'], c=core['cluster_label'], cmap='tab10', s=25, alpha=0.8)

# Mark centroids
if not df_summary.empty:
    plt.scatter(df_summary['centroid_longitude'], df_summary['centroid_latitude'], c='red', marker='X', s=120, edgecolors='black', label='Hotspot Centroids')

plt.title('Mumbai Road Incident DBSCAN Hotspots (Eps=1.2 km)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()
